# Results: NYU Depth v2 regression

Builds the depth-regression results table (LaTeX + CSV in `notebooks/tables/`) and the
qualitative depth figures of the appendix (`notebooks/figures/paper/`) from the trained runs:

1. **Results table** (mean ± std over seeds) → LaTeX + CSV.
2. **Qualitative appendix figures**: mean / error / marginal std / predictive samples per method.
3. **Anchor-pixel covariance maps**: model-implied covariance vs. the empirical
   covariance of test residuals. They double as a check that the LorD low-rank factor is
   active: with a zero-initialized `other_proj`, `dL/dU ∝ U` vanishes at `U = 0` and the
   factor would never move.

**Prerequisites.** Checkpoints under `logs/train/runs/`, the NYU Depth v2 data, the
DINOv2 / Depth Anything V2 checkout, and a W&B login. Headless:
`jupyter nbconvert --execute --to notebook --inplace notebooks/paper_depth_results.ipynb`.

In [ ]:
import os
import sys
from collections import defaultdict
from pathlib import Path
from statistics import mean, stdev

# Repo root via the .project-root marker so `import src...` works from notebooks/.
REPO = next(p for p in Path.cwd().resolve().parents if (p / ".project-root").exists())
sys.path.insert(0, str(REPO))
os.environ.setdefault("PROJECT_ROOT", str(REPO))

import hydra
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb
from omegaconf import OmegaConf
from src.utils.legacy import upgrade_legacy_targets

from src.models.gaussian_predictive import LightningPredictiveModel

WANDB_PROJECT = "precip/spectral-experiments"
WANDB_USER = "emil-partow"
CAMPAIGN_START = "2026-08-03T09:00"   # ignore runs logged before this time

RUNS_DIR = REPO / "logs" / "train" / "runs"
FIG_DIR = REPO / "notebooks" / "figures" / "paper"
TAB_DIR = REPO / "notebooks" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42                    # seed whose checkpoints feed the figures
TEST_IDX = [7, 411, 1900]    # fixed NYU test images for the qualitative grids
N_SAMPLES = 2                # predictive samples per method
CMAP_DEPTH, CMAP_STD, CMAP_ERR = "magma", "Blues", "RdBu_r"

## 0. Index the run directories

Hydra names its run dir by timestamp at second resolution, so SLURM array tasks that
start together **share one directory** and interleave their checkpoints and configs.
Such directories are detected here (more than one `wandb/run-*` inside) and excluded
from checkpoint discovery -- their checkpoints cannot be attributed to a config.

Two indices come out of this:

* `cfg_by_wandb` -- W&B run id → that run's Hydra config. Needed because W&B only
  records the *Lightning module's* arguments (`save_hyperparameters`), so the
  method-level sweep knobs (`temperature`, `normalize_core`) exist only on disk.
* `ckpt_index` -- (method variant, seed) → newest usable checkpoint. Later run dirs
  win, so a rerun supersedes the run it replaced.

In [ ]:
def method_key(cfg):
    """Variant label used to match checkpoints; mirrors the table's row labels."""
    m = cfg.method
    target = m._target_.split(".")[-1]
    if target != "SCORE":
        return f"{target}_{m.loss}"
    parts = []
    if m.get("stage2_loss") != "kernel_score":
        parts.append(f"s2={m.get('stage2_loss')}")
    if m.get("freeze_backbone", True) is False:
        parts.append("finetune")
    if float(m.get("temperature", 1.0)) != 1.0:
        parts.append(f"T={m.get('temperature')}")
    if m.get("stage1_loss") != "marginal_kernel_score":
        parts.append(f"s1={m.get('stage1_loss')}")
    if m.get("normalize_core", True) is False:
        parts.append("free-core")
    return "Whitened[" + (",".join(parts) if parts else "default") + "]"


cfg_by_wandb, ckpt_index, collided = {}, {}, []
for d in sorted(RUNS_DIR.iterdir()):
    cfg_file = d / ".hydra" / "config.yaml"
    if not cfg_file.exists():
        continue
    try:
        cfg = upgrade_legacy_targets(OmegaConf.load(cfg_file))
    except Exception:
        collided.append(d.name)   # config.yaml shredded by concurrent writers
        continue
    if "DepthDataModule" not in str(cfg.datamodule._target_):
        continue

    wandb_dirs = sorted(d.glob("wandb/run-*"))
    for wb in wandb_dirs:
        cfg_by_wandb[wb.name.split("-")[-1]] = cfg
    if len(wandb_dirs) > 1:
        collided.append(d.name)   # several jobs shared this dir -> unattributable
        continue

    ckpt_dir = d / "checkpoints"
    if (ckpt_dir / "stage2").is_dir():
        ckpt_dir = ckpt_dir / "stage2"       # SCORE: stage-2 checkpoint (fitted core)
    ckpts = sorted(ckpt_dir.glob("*.ckpt"), key=lambda p: p.stat().st_mtime) \
        if ckpt_dir.is_dir() else []
    if ckpts:
        # Later dirs overwrite earlier ones: a rerun supersedes what it replaced.
        ckpt_index[(method_key(cfg), int(cfg.seed))] = (ckpts[-1], cfg)

print(f"{len(ckpt_index)} checkpoints indexed, {len(collided)} collided dirs skipped")
for key in sorted(ckpt_index):
    print("  ", key, "->", ckpt_index[key][0].relative_to(RUNS_DIR))

## 1. Results table (mean ± std over seeds)

W&B is the primary source -- one entry per run, carrying the seed and the
`evaluation/test_*` metrics. The disk configs fill in only what W&B cannot know
(`temperature`, `normalize_core`). Without that join every sweep variant would
collapse into the single default SCORE row (`whitened [default]`).

The seed column is printed deliberately: it makes a missing or duplicated run
visible instead of silently shrinking a mean.

In [ ]:
METRICS = ["mse", "mae", "crps", "energy_score", "kernel_score", "nll"]
ROW_ORDER = [
    "deterministic_mse",
    "phys_diag_nll",
    "phys_diag_kernel_score",
    "phys_lora_kernel_score",
    "spectral_diag_kernel_score",
    "whitened [default]",
    "whitened [s2=nll]",
    "whitened [s1=crps]",
    "whitened [free-core]",
    "whitened [T=0.5]",
    "whitened [T=2.0]",
    "whitened [T=4.0]",
    "whitened [finetune]",
]


def run_label(run):
    """Row label. Stage losses / freezing come from W&B, the method-level sweep
    knobs from the run's Hydra config (see the index cell)."""
    name = run.name.replace("depth_", "")
    if "whitened" not in name:
        return name
    c = run.config
    m = cfg_by_wandb[run.id].method if run.id in cfg_by_wandb else {}
    parts = []
    if c.get("stage2_loss", "kernel_score") != "kernel_score":
        parts.append(f"s2={c.get('stage2_loss')}")
    if c.get("freeze_backbone", True) is False:
        parts.append("finetune")
    if m and float(m.get("temperature", 1.0)) != 1.0:
        parts.append(f"T={m.get('temperature')}")
    if c.get("stage1_loss", "marginal_kernel_score") != "marginal_kernel_score":
        parts.append(f"s1={c.get('stage1_loss')}")
    if m and m.get("normalize_core", True) is False:
        parts.append("free-core")
    return "whitened [" + (", ".join(parts) if parts else "default") + "]"


api = wandb.Api()
summaries, seeds_seen = defaultdict(list), defaultdict(list)
for run in api.runs(WANDB_PROJECT, order="-created_at"):
    user = run.user.username if run.user else "?"
    if user != WANDB_USER or str(run.created_at) < CAMPAIGN_START:
        continue
    if not run.name.startswith("depth_"):
        continue
    summary = dict(run.summary)
    if not any(k.startswith("evaluation/") for k in summary):
        continue        # crashed before eval, or still training
    label = run_label(run)
    summaries[label].append(summary)
    seeds_seen[label].append(run.config.get("seed"))


def fmt(values):
    values = [v for v in values if isinstance(v, (int, float))]
    if not values:
        return "--"
    m = mean(values)
    sd = stdev(values) if len(values) > 1 else 0.0
    return f"{m:.3f} \\pm {sd:.3f}"


rows = {}
for label in ROW_ORDER + [k for k in summaries if k not in ROW_ORDER]:
    if label not in summaries:
        continue
    ss = summaries[label]
    seeds = ",".join(str(s) for s in sorted(x for x in seeds_seen[label] if x is not None))
    rows[label] = {"n": len(ss), "seeds": seeds,
                   **{k: fmt([s.get(f"evaluation/test_{k}") for s in ss]) for k in METRICS}}

df = pd.DataFrame(rows).T
df.to_csv(TAB_DIR / "depth_results.csv")
(TAB_DIR / "depth_results.tex").write_text(
    df.drop(columns=["seeds"]).to_latex(column_format="l" + "r" * (len(METRICS) + 1),
                                        escape=False)
)
print(f"{sum(r['n'] for r in rows.values())} runs total -> {TAB_DIR / 'depth_results.tex'}")
df

## 2. Qualitative appendix figures

Per test image, one row per method: **mean / signed error / marginal std / predictive
samples**, with input RGB and ground truth in the header row. All depth panels share
one color scale, the std maps share another, and the error map is symmetric about
zero -- so differences between methods are real, not colormap artifacts.

Det (`deterministic`) is omitted here: a Dirac predictive has zero std and samples equal to
its mean, so its columns carry no information.

In [ ]:
SHOWN = [
    ("PhysDiag_nll", "Diag (NLL)"),
    ("PhysDiag_kernel_score", "Diag (KS)"),
    ("PhysLoRa_kernel_score", "LorD (KS)"),
    ("SpectralDiag_kernel_score", "Spectral (KS)"),
    ("Whitened[default]", "SCORE (KS)"),
    ("Whitened[s2=nll]", "SCORE (NLL)"),
]


def load_model(ckpt, cfg):
    """Rebuild method/backbone from the run's config, then restore the weights
    (mirrors src/evaluation/evaluate.py; gamma is irrelevant for predictions)."""
    method = hydra.utils.instantiate(cfg.method, out_channels=1, gamma=1.0, gamma_eval=1.0)
    backbone = hydra.utils.instantiate(
        cfg.backbone,
        in_channels=3,
        out_channels=method.out_channels,
        group_widths=method.group_widths,
        group_denorm=method.group_denorm,
        other_zero_init=method.other_zero_init,
    )
    model = LightningPredictiveModel.load_from_checkpoint(
        str(ckpt), backbone=backbone, predictive_method=method, map_location="cpu"
    )
    model.eval()
    return model, method


cfg_any = next(iter(ckpt_index.values()))[1]
dm = hydra.utils.instantiate(
    OmegaConf.merge(cfg_any.datamodule, {"num_workers": 0, "preload": False})
)
dm.setup("test")
X = torch.stack([dm.test_dataset[i][0] for i in TEST_IDX])
Y = torch.stack([dm.test_dataset[i][1] for i in TEST_IDX])

predictions = {}
for key, display in SHOWN:
    if (key, SEED) not in ckpt_index:
        print(f"missing checkpoint: {key} seed {SEED} -- skipped")
        continue
    ckpt, cfg = ckpt_index[(key, SEED)]
    model, method = load_model(ckpt, cfg)
    with torch.no_grad():
        pred = model(X)
        predictions[display] = (
            pred[..., 0].numpy(),
            method.get_marg_std(pred).numpy(),
            method.get_samples(pred, n_samples=N_SAMPLES).numpy(),
        )
    print(f"predicted: {display}  ({ckpt.relative_to(RUNS_DIR)})")

col_titles = ["Mean", "Error", "Std"] + [f"Sample {i + 1}" for i in range(N_SAMPLES)]
n_rows, n_cols = 1 + len(predictions), 3 + N_SAMPLES
for b, test_i in enumerate(TEST_IDX):
    gt = Y[b, 0].numpy()
    vmin, vmax = np.percentile(gt, 1), np.percentile(gt, 99)
    err_max = max(np.percentile(np.abs(m[b, 0] - gt), 99) for m, _, _ in predictions.values())
    std_max = max(np.percentile(s[b, 0], 99) for _, s, _ in predictions.values())

    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(2.1 * n_cols, 1.75 * n_rows), constrained_layout=True
    )
    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
    axes[0, 0].imshow(X[b].permute(1, 2, 0).numpy())
    axes[0, 0].set_title("Input RGB", fontsize=9)
    im_depth = axes[0, 1].imshow(gt, cmap=CMAP_DEPTH, vmin=vmin, vmax=vmax)
    axes[0, 1].set_title("Ground truth", fontsize=9)
    for c in range(2, n_cols):
        axes[0, c].axis("off")

    for r, (display, (mean_map, std_map, samples)) in enumerate(predictions.items(), start=1):
        axes[r, 0].imshow(mean_map[b, 0], cmap=CMAP_DEPTH, vmin=vmin, vmax=vmax)
        im_err = axes[r, 1].imshow(mean_map[b, 0] - gt, cmap=CMAP_ERR,
                                   vmin=-err_max, vmax=err_max)
        im_std = axes[r, 2].imshow(std_map[b, 0], cmap=CMAP_STD, vmin=0, vmax=std_max)
        for s_i in range(N_SAMPLES):
            axes[r, 3 + s_i].imshow(samples[b, 0, :, :, s_i], cmap=CMAP_DEPTH,
                                    vmin=vmin, vmax=vmax)
        axes[r, 0].set_ylabel(display, fontsize=9)
        if r == 1:
            for c, title in enumerate(col_titles):
                axes[r, c].set_title(title, fontsize=9)

    fig.colorbar(im_depth, ax=axes[:, 0], shrink=0.5, label="depth", location="left")
    fig.colorbar(im_err, ax=axes[:, 1], shrink=0.4, location="bottom", label="error")
    fig.colorbar(im_std, ax=axes[:, 2], shrink=0.4, location="bottom", label="std")
    fig.suptitle(f"NYU test sample {test_i} (seed {SEED})", fontsize=10)
    out = FIG_DIR / f"qualitative_test{test_i}_seed{SEED}.png"
    fig.savefig(out, dpi=200)
    plt.show()
    print("saved:", out)

## 3. Anchor-pixel covariance maps -- and the low-rank check

For a few anchor pixels: the **unnormalized** covariance of that anchor with every
other pixel (per-row color scale). The ground-truth column is the empirical covariance
of test residuals `y - mu`; the model columns use each method's closed-form covariance row.

Correlation is deliberately *not* shown: normalizing divides out the per-pixel scale
`D`, under which every circulant model looks stationary and featureless. In covariance
`D` modulates the map, the scene geometry appears, and the models become comparable to
the empirical reference.

**This figure also checks that the LorD low-rank factor is alive.** A zero-initialized
`other_proj` gives `dL/dU ∝ U = 0`, so the factor would stay identically zero
(`max|off| = 0.0`). The printed diagnostics make that quantitative: any method other than
Diag (diagonal by construction) whose off-diagonal covariance is exactly `0.0e+00` has a
dead head.

In [ ]:
COV_TEST_I = 411
ANCHOR_PIXELS = [(28, 44), (64, 80), (100, 132)]  # (y, x)
N_EMP = 512                                        # test images for the empirical cov
COV_SHOWN = [
    ("PhysDiag_kernel_score", "Diag (KS)"),
    ("PhysLoRa_kernel_score", "LorD (KS)"),
    ("SpectralDiag_kernel_score", "Spectral (KS)"),
    ("Whitened[default]", "SCORE (KS)"),
    ("Whitened[s2=nll]", "SCORE (NLL)"),
]


def model_cov_map(key, pred, anchor):
    """Model covariance of the anchor with every pixel (Fig. 4b formulas)."""
    ay, ax_ = anchor
    diag_or_s = pred[0, 0, :, :, 1].double()
    if key.startswith("Diag"):
        cov = torch.zeros_like(diag_or_s)
        cov[ay, ax_] = diag_or_s[ay, ax_]          # diagonal: no off-diagonal structure
        return cov.numpy()
    if key.startswith("LorD"):
        u = pred[0, 0, :, :, 2:].double()
        cov = (u * u[ay, ax_, :]).sum(dim=-1).clone()
        cov[ay, ax_] = cov[ay, ax_] + diag_or_s[ay, ax_]
        return cov.numpy()
    if key.startswith("SpectralDiag"):
        # Sigma = F^H diag(s) F: the covariance row is the circulant autocovariance.
        c = torch.fft.ifft2(diag_or_s).real
        return torch.roll(c, shifts=(ay, ax_), dims=(0, 1)).numpy()
    # SCORE: Sigma = D C D with c_core(0) = mean(s) = 1 (scale gauge of the core).
    s = pred[0, 0, :, :, 2].double()
    c_core = torch.fft.ifft2(s).real
    d_half = diag_or_s.sqrt()
    return (d_half * torch.roll(c_core, shifts=(ay, ax_), dims=(0, 1))
            * d_half[ay, ax_]).numpy()


# Ground truth: covariance of the test residuals. mu comes from the SCORE model;
# the methods' means are near-identical, so one shared reference is enough.
ckpt, cfg = ckpt_index[("Whitened[default]", SEED)]
mean_model, _ = load_model(ckpt, cfg)
residuals = []
with torch.no_grad():
    for start in range(0, N_EMP, 16):
        batch = range(start, min(start + 16, N_EMP))
        xb = torch.stack([dm.test_dataset[i][0] for i in batch])
        yb = torch.stack([dm.test_dataset[i][1] for i in batch])
        residuals.append(yb[:, 0] - mean_model(xb)[..., 0][:, 0])
R = torch.cat(residuals).double()
R = R - R.mean(dim=0, keepdim=True)


def empirical_cov(anchor):
    ay, ax_ = anchor
    return (R * R[:, ay, ax_][:, None, None]).mean(dim=0).numpy()


x_cov, y_cov = dm.test_dataset[COV_TEST_I]
cov_maps = {}
for key, display in COV_SHOWN:
    ckpt, cfg = ckpt_index[(key, SEED)]
    m, _ = load_model(ckpt, cfg)
    with torch.no_grad():
        pred = m(x_cov.unsqueeze(0))
    cov_maps[display] = {a: model_cov_map(key, pred, a) for a in ANCHOR_PIXELS}

# --- Diagnostics: magnitude and radial decay for the middle anchor ---
A = ANCHOR_PIXELS[1]
ay, ax_ = A
H, W = R.shape[1], R.shape[2]
yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
off = ~((yy == ay) & (xx == ax_))
dist = np.sqrt((yy - ay) ** 2 + (xx - ax_) ** 2)
RINGS = [(0.5, 1.5), (1.5, 3), (3, 6), (6, 12), (12, 24), (24, 48), (48, 200)]
gt_cov = empirical_cov(A)

print(f"=== covariance magnitude (anchor {A}) ===")
print(f"{'method':22s}{'max|cov_off|':>14s}{'cov[a,a]':>13s}{'off/self':>11s}")
print(f"{'ground truth':22s}{np.abs(gt_cov[off]).max():14.3e}{gt_cov[ay, ax_]:13.3e}"
      f"{np.abs(gt_cov[off]).max() / gt_cov[ay, ax_]:11.4f}")
for _, display in COV_SHOWN:
    c = cov_maps[display][A]
    mx, self_c = float(np.abs(c[off]).max()), float(c[ay, ax_])
    dead = "   <-- DEAD HEAD" if mx == 0.0 and not display.startswith("Diag") else ""
    print(f"{display:22s}{mx:14.3e}{self_c:13.3e}"
          f"{(mx / self_c if self_c else float('nan')):11.4f}{dead}")

print(f"\n=== covariance averaged over distance rings (anchor {A}) ===")
print(f"{'method':22s}" + "".join(f"{f'{lo:g}-{hi:g}px':>11s}" for lo, hi in RINGS))


def ring_profile(m):
    return "".join(
        f"{float(np.mean(m[(dist >= lo) & (dist < hi)])):11.2e}" for lo, hi in RINGS
    )


print(f"{'ground truth':22s}" + ring_profile(gt_cov))
for _, display in COV_SHOWN:
    print(f"{display:22s}" + ring_profile(cov_maps[display][A]))

# --- Figure ---
rgb = x_cov.permute(1, 2, 0).numpy()
gt_depth = y_cov[0].numpy()
model_cols = [display for _, display in COV_SHOWN]
ncols = 3 + len(model_cols)
fig, axes = plt.subplots(
    len(ANCHOR_PIXELS), ncols,
    figsize=(2.5 * ncols, 2.1 * len(ANCHOR_PIXELS)), constrained_layout=True,
)
for r_i, anchor in enumerate(ANCHOR_PIXELS):
    axes[r_i, 0].imshow(rgb)
    axes[r_i, 0].set_ylabel(f"anchor {anchor}", fontsize=8)
    axes[r_i, 1].imshow(gt_depth, cmap=CMAP_DEPTH,
                        vmin=np.percentile(gt_depth, 1), vmax=np.percentile(gt_depth, 99))
    cov_list = [empirical_cov(anchor)] + [cov_maps[d][anchor] for d in model_cols]
    # Row scale anchored at the row's largest |cov|, excluding the self-covariance
    # peak (it would drown all off-diagonal structure). A symlog scale instead
    # saturates the large panels to uniform red, so the magnitudes are annotated.
    vmax = max(
        float(np.percentile(np.abs(np.where(m == m.max(), 0, m)), 99.9)) for m in cov_list
    )
    for c_i, m in enumerate(cov_list, start=2):
        pcm = axes[r_i, c_i].pcolormesh(m, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                                        rasterized=True)
        axes[r_i, c_i].invert_yaxis()
        off_m = m.copy()
        off_m[anchor[0], anchor[1]] = 0.0
        axes[r_i, c_i].text(0.02, 0.02, f"max|off|={np.abs(off_m).max():.1e}",
                            transform=axes[r_i, c_i].transAxes, fontsize=6,
                            color="#333333", va="bottom")
    for c_i in range(ncols):
        ax = axes[r_i, c_i]
        ax.plot([anchor[1]], [anchor[0]], "kx", ms=6, mew=1.5)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(pcm, ax=list(axes[r_i, 2:]), fraction=0.008, pad=0.01,
                 label=f"cov (+-{vmax:.1e})")
for c_i, title in enumerate(
    ["input RGB", "ground truth depth", "ground truth cov (empirical)"] + model_cols
):
    axes[0, c_i].set_title(title, fontsize=8)
fig.suptitle(
    f"Covariance with the anchor pixel (x): ground truth (empirical test residuals, "
    f"N={N_EMP}) vs. model-implied -- test {COV_TEST_I}, seed {SEED}"
)
out = FIG_DIR / f"covariance_anchor_maps_seed{SEED}.png"
fig.savefig(out, dpi=150)
plt.show()
print("saved:", out)

### Reading the results

* **LorD vs. Diag.** With a live low-rank head, the rank-1 factor `U` in
  `Sigma = diag(d) + U Uᵀ` contributes a real, if modest, gain over Diag (test NLL
  improves, energy score slightly). Rank 1 plus a shape-blind kernel score limits how much
  it can add; `rank > 1` or NLL training are the natural next knobs.
* **Spectral (KS)** (`spectral_diag`, a circulant covariance in physical space) has a
  *constant* std map by construction -- a circulant covariance has constant marginal
  variance (Parseval). Its off-diagonal covariance is nonzero but ~20x smaller than the
  ground truth and decays within ~2 px: its learned spectrum stays nearly white
  (max/min ≈ 2, versus ~4e+04 for the SCORE core), so it captures the scale but not the shape.
* **SCORE** is the only model whose covariance carries per-pixel scale *and*
  spatial correlation, so the scene geometry appears in its covariance maps. The
  stage-2 loss decides the range: the kernel score gives a broad kernel that tracks
  the ground-truth decay well beyond ~3 px but stays too flat far away; NLL is sharply
  concentrated -- closest at the shortest lag but ~0 beyond ~20 px, missing the
  long-range component the data has. NLL nonetheless attains the far better test
  likelihood *and* is two orders of magnitude more stable across seeds, so the two
  scores disagree about what matters.
* **Ablations.** `stage1_loss=crps` gives the best CRPS in the whole table; dropping
  the core normalization (`free-core`) clearly hurts the likelihood; temperature
  scaling of the core changes nothing beyond seed noise (T only rescales the
  log-spectrum globally and cannot reorder modes); stage-2 full finetuning diverges.
* **Diag (NLL)** vs **Diag (KS)** in the qualitative grids: same model class,
  different loss -- the NLL std map is visibly mis-scaled (collapse onto the quantized
  depth targets), matching its worse test scores.